[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/halla-ai/deepnlp-2026/blob/main/notebooks/week-05.ipynb)

# 5주차 실습: 평가 시스템과 지표가 측정하지 못하는 것

**목표.** 4주차의 제주 관광 문의 채점셋을 그대로 써서 **규칙 채점(정답지 비교)과 judge 채점(문서만 준 판정)**을 대조하고 불일치를 기록한다. 그다음 pass@k를 측정해 **k 곡선**을 그리고, 성능 향상이 새 추론 능력인지 탐색 효율 개선인지 판정한다.

강의 노트 5주차의 false acceptance와 pass@k 내용과 같은 맥락이다. judge도 다음 토큰 확률 기계라는 점에 주목하자.

## 0. 준비

아래 셀을 실행해 필요한 라이브러리를 설치한다. GPU는 필요 없다.

In [ ]:
# 필요한 것 설치 (Colab에서 한 번만)
!pip -q install transformers torch

## 1. 먼저 그냥 실행해 보기

아래 셀들을 위에서부터 차례로 실행하세요. 아무것도 고치지 않아도 끝까지 돌아갑니다.

### 1-1. 채점셋 - 4주차의 제주 관광 문의 24건을 그대로

4주차와 같은 채점셋이다. 라벨은 주차, 시설, 요금 세 가지다.

In [ ]:
# 제주 관광 문의 채점셋 (4주차와 동일)
eval_set = [
    ("성산일출봉 주차장이 어디예요?", "주차"),
    ("함덕해수욕장에 주차할 곳이 있나요?", "주차"),
    ("제주공항에서 렌터카를 어디서 반납하나요?", "주차"),
    ("만장굴 주차 요금이 있나요?", "주차"),
    ("협재해수욕장 주차장이 만차인지 알 수 있나요?", "주차"),
    ("한라산 어리목 탐방로 주차가 가능한가요?", "주차"),
    ("천지연폭포 주차장에서 입구까지 멀어요?", "주차"),
    ("섭지코지 주차 공간이 넓은가요?", "주차"),
    ("이호테우해수욕장에 샤워실이 있나요?", "시설"),
    ("성산일출봉에 화장실이 많이 있나요?", "시설"),
    ("함덕해수욕장에 파라솔을 빌릴 수 있나요?", "시설"),
    ("제주민속촌에 수유실이 있나요?", "시설"),
    ("한라산 국립공원에 매점이 있나요?", "시설"),
    ("월정리해수욕장에 짐 보관함이 있나요?", "시설"),
    ("만장굴 안을 휠체어가 다닐 수 있나요?", "시설"),
    ("식물원에 유모차 대여가 되나요?", "시설"),
    ("성산일출봉 입장료가 얼마예요?", "요금"),
    ("만장굴 입장료 할인이 있나요?", "요금"),
    ("제주민속촌 가족권 가격이 어떻게 되나요?", "요금"),
    ("식물원 입장권을 온라인으로 사면 더 싼가요?", "요금"),
    ("우도 왕복 배 삯이 얼마인가요?", "요금"),
    ("한라산 트레킹은 무료인가요?", "요금"),
    ("청소년은 입장료가 할인되나요?", "요금"),
    ("오름 이용 요금이 따로 있나요?", "요금"),
]

LABELS = ["주차", "시설", "요금"]
print(f"채점셋: {len(eval_set)}건")
for lab in LABELS:
    print(f"  {lab}: {sum(1 for _, y in eval_set if y == lab)}건")

### 1-2. 모델 로딩 - 가중치는 그대로

작은 한국어 GPT 모델을 쓴다. 학습시키지 않는다. 난수도, 데이터로의 갱신도 없이, 다운로드한 가중치 그대로다.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-0.6B-Base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

total = sum(p.numel() for p in model.parameters())
print(f"전체 파라미터: {total:,}")
print("이번 주에 갱신하는 파라미터: 0 (모델은 그대로, 채점 방식만 비교한다)")

### 1-3. 라벨 확률로 분류하기 - 4주차의 규칙 채점

프롬프트 바로 다음 토큰 자리의 확률분포에서 각 라벨의 첫 토큰 확률을 비교해 분류한다. 이것이 **규칙 채점**의 기준선이다. 정답지와 비교하므로 false acceptance가 없다. 대신 정답이 있는 문항밖에 채점할 수 없다.

In [ ]:
import torch

def label_scores(prompt_text):
    # 프롬프트 바로 다음 토큰 자리에서 각 라벨 첫 토큰의 로그확률을 비교한다
    prefix_ids = tokenizer(prompt_text, return_tensors="pt").input_ids
    with torch.no_grad():
        logits = model(prefix_ids).logits
    next_token_logps = torch.log_softmax(logits[0, -1], dim=-1)
    return {lab: next_token_logps[tokenizer(" " + lab, add_special_tokens=False).input_ids[0]].item()
            for lab in LABELS}

PROMPT_A = """아래 제주 관광 문의를 주차, 시설, 요금 중 하나로 분류하시오.
라벨만 답하시오.

문의: {query}
라벨:"""

# 동작 확인: 4주차와 같은 문의로 한 건 맞혀 보기
trial = PROMPT_A.format(query="섭지코지 주차 공간이 넓은가요?")
trial_scores = label_scores(trial)
for lab, sc in trial_scores.items():
    print(f"  {lab}: {sc:.3f}")
print("-> 모델의 선택:", max(trial_scores, key=trial_scores.get))

### 1-4. 규칙 채점 전체 실행 - 정답지 비교

채점셋 전체를 돌려 정확도를 매긴다. **정답지와 비교하므로 틀린 것은 틀렸다고 나온다.** judge 채점과 대조할 기준선이 된다.

In [ ]:
def rule_evaluate():
    results = []
    for sentence, gold in eval_set:
        scores = label_scores(PROMPT_A.format(query=sentence))
        pred = max(scores, key=scores.get)
        results.append((sentence, gold, pred))
    return results

rule_results = rule_evaluate()
rule_correct = sum(1 for _, gold, pred in rule_results if gold == pred)
print(f"규칙 채점 정확도: {rule_correct}/{len(eval_set)} = {rule_correct/len(eval_set):.2f}")
print()
print("틀린 문항:")
for sentence, gold, pred in rule_results:
    if gold != pred:
        print(f"  [틀림] {sentence}")
        print(f"         정답: {gold} / 모델: {pred}")

### 1-5. judge 채점 - 문서만 준 판정 프롬프트

이제 judge 채점이다. 모델에게 **루브릭(채점 기준표) 문서만** 주고, 모델 출력이 맞는지 판정하게 한다. **정답지는 주지 않는다.** 강의 노트의 false acceptance 실험과 같은 설정이다.

judge 판정도 다음 토큰 확률 비교다. "맞음"과 "틀림" 토큰 중 어느 쪽에 높은 확률을 두는지 본다.

In [ ]:
# judge 판정 프롬프트. 정답지는 넣지 않는다. 문서(루브릭)만 준다
JUDGE_PROMPT = """[역할] 당신은 분류 채점자입니다.
아래의 문의와 모델의 분류 출력을 보고, 출력이 문의를 정확히 분류했는지 판정하시오.
답은 "맞음" 또는 "틀림"으로만 쓰시오.

[루브릭]
- 출력의 라벨이 문의의 핵심 요구에 부합하면 "맞음"
- 출력의 라벨이 문의의 핵심 요구와 어긋나면 "틀림"

[문의] {query}
[모델 출력] 라벨: {pred}

[판정]"""

# "맞음"과 "틀림"의 첫 토큰 확률을 비교해 판정을 읽는다
JUDGE_LABELS = ["맞음", "틀림"]

def judge_scores(prompt_text):
    prefix_ids = tokenizer(prompt_text, return_tensors="pt").input_ids
    with torch.no_grad():
        logits = model(prefix_ids).logits
    next_token_logps = torch.log_softmax(logits[0, -1], dim=-1)
    return {lab: next_token_logps[tokenizer(" " + lab, add_special_tokens=False).input_ids[0]].item()
            for lab in JUDGE_LABELS}

def judge_verdict(query, pred):
    sc = judge_scores(JUDGE_PROMPT.format(query=query, pred=pred))
    return max(sc, key=sc.get)

# 동작 확인: 규칙 채점에서 맞은 문항 하나를 judge에게도 물어 본다
trial_sentence, trial_gold, trial_pred = rule_results[0]
print(f"문의: {trial_sentence}")
print(f"정답: {trial_gold} / 모델: {trial_pred}")
print(f"judge 판정: {judge_verdict(trial_sentence, trial_pred)}")

### 1-6. 규칙 채점 vs judge 채점 대조 - 불일치 기록

이제 채점셋 전체를 두 방식으로 채점하고 나란히 놓는다. judge는 **모델의 출력(예측)이 문의에 부합하는가**만 보고 판정하며, 정답지를 모른다.

judge가 "맞음"이라는데 정답지 기준으로는 틀린 문항이 바로 **false acceptance** 사례다.

In [ ]:
print(f"{'문의':<38} {'정답':<5} {'모델':<5} {'judge':<5} 불일치")
print("-" * 75)
mismatch_count = 0
false_accept_count = 0
for sentence, gold, pred in rule_results:
    verdict = judge_verdict(sentence, pred)
    rule_ok = (gold == pred)
    judge_ok = (verdict == "맞음")
    mismatch = "<-- 불일치" if rule_ok != judge_ok else ""
    if rule_ok != judge_ok:
        mismatch_count += 1
    if judge_ok and not rule_ok:
        false_accept_count += 1
    mark = "정답" if rule_ok else "오답"
    print(f"{sentence[:36]:<38} {gold:<5} {pred:<5} {verdict:<5} {mismatch}")

print()
print(f"규칙 채점과 judge 채점의 불일치: {mismatch_count}건 / {len(eval_set)}건")
print(f"그중 judge가 오답을 승인한 사례(false acceptance): {false_accept_count}건")
print()
print("** 수집된 오답 승인 사례 (학습활동에 그대로 쓴다) **")
for sentence, gold, pred in rule_results:
    verdict = judge_verdict(sentence, pred)
    if verdict == "맞음" and gold != pred:
        print(f"  문의: {sentence}")
        print(f"  정답: {gold} / 모델 출력: {pred} / judge 판정: 맞음")
        print(f"  -> judge는 근거 문장의 형식만 보고 '맞음'을 골랐다. 정답지와 비교하지 않아 틀림을 못 잡는다")
        print()

### 1-7. pass@k - k번의 시도 안에 맞히는 비율

이제 **pass@k**를 측정한다. 분류 과제에서는 한 문항을 k번 시도해 그중 한 번이라도 정답 라벨을 내는 비율이다.

무편 추정량은 강의 노트와 같다: pass@k = 1 - C(n-c, k) / C(n, k). n은 총 시행 횟수, c는 그중 정답 시행 수다.

분류 과제이므로 생성 샘플링 대신 **라벨 확률분포에서 k번 다항 샘플링**으로 시행을 흉내 낸다. n=20이면 한 문항당 20번 시도한 것과 같다. GPU도, 생성도 필요 없다.

In [ ]:
import math
from collections import Counter

torch.manual_seed(42)  # 재현성을 위한 시드. 바꾸면 곡선이 조금 흔들린다

N_SAMPLES = 20  # 문항당 시행 횟수 n
K_LIST = [1, 2, 4, 8, 16]

def pass_at_k(c, n, k):
    # 무편 추정량: 1 - C(n-c, k) / C(n, k)
    if k > n - c:
        return 1.0
    return 1.0 - math.comb(n - c, k) / math.comb(n, k)

def pass_at_k_curve(prompt, n=N_SAMPLES):
    # 채점셋 전체의 pass@k를 추정한다
    k_vals = K_LIST
    totals = {k: 0.0 for k in k_vals}
    for sentence, gold in eval_set:
        scores = label_scores(prompt.format(query=sentence))
        labs = list(scores.keys())
        logps = torch.tensor([scores[lab] for lab in labs])
        probs = torch.softmax(logps, dim=0).numpy()
        gold_idx = labs.index(gold)
        # n번 다항 샘플링
        samples = torch.multinomial(torch.tensor(probs, dtype=torch.float), n, replacement=True).tolist()
        c = sum(1 for s in samples if s == gold_idx)
        for k in k_vals:
            totals[k] += pass_at_k(c, n, k)
    return {k: totals[k] / len(eval_set) for k in k_vals}

curve_before = pass_at_k_curve(PROMPT_A)
print(f"{'k':<6} {'pass@k (학습 전)'}")
for k in K_LIST:
    print(f"{k:<6} {curve_before[k]:.3f}")

## 2. 한 지점만 바꿔 보기 - k 리스트를 연장해 곡선을 다시 그린다

아래 셀의 `# TODO` 로 표시된 **한 곳만** 바꾸고 다시 실행하세요.

> 규칙: `K_LIST` 를 연장해 k 곡선을 촘촘하게 만든다. 바꾸기 전 곡선을 적어 두면 비교할 수 있습니다. k를 크게 하면 곡선이 어디서 **천장**에 닿는지 확인하고, 학습 전 pass@1이 낮게 나온 것과 대조해 천장이 높아지는가(새 능력) 또는 기울기만 상승하는가(탐색 효율)를 문장으로 해석해 본다.

In [ ]:
# TODO: K_LIST를 연장해 곡선을 다시 그린다. 예: [1, 2, 4, 8, 16, 18, 20]
K_LIST = [1, 2, 4, 8, 16]

# 아래는 그대로 둡니다
curve_before = pass_at_k_curve(PROMPT_A)
print(f"{'k':<6} {'pass@k (학습 전)'}")
for k in K_LIST:
    print(f"{k:<6} {curve_before[k]:.3f}")

## 3. 곡선 그리기

수집한 pass@k 값을 matplotlib으로 그린다. 강의 노트의 곡선 그림(학습 전 천장 0.30, 학습 후 천장 0.38)은 **설명용 예시**라 여기서 나오는 곡선과 다를 수 있다. 볼 것은 **곡선이 어디서 천장에 닿는가**다.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4.5))
plt.plot(K_LIST, [curve_before[k] for k in K_LIST], "o-", color="#e03131", label="학습 전 pass@k (PROMPT_A)")
plt.xlabel("k (문항당 시도 횟수)")
plt.ylabel("pass@k")
plt.title("pass@k 곡선 (설명용 예시가 아니라 실측)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

### 3-1. judge 채점을 pass@k 관점에서 다시 보기

judge 채점은 사실 k=1인 pass@k와 같다. 문항당 한 번 판정을 뽑고 그것이 "맞음"인지 보니까. 그래서 judge 채점의 false acceptance는 **k를 늘리면 잡을 수 있는가**를 물어 볼 수 있다.

k를 늘려 판정을 여러 번 뽑아 과반 투표하면 false acceptance가 줄어드는가 보자. (이 부분은 관찰용이다. 실제 운영에서는 판정을 k번 뽑는 비용이 든다)

In [ ]:
# judge 판정을 k=5번 뽑아 과반 투표하면 false acceptance가 줄어드는가
# (확률적 시뮬레이션. label_scores 대신 judge_scores를 쓴다)
J_K = 5

def judge_majority(query, pred, k=J_K):
    sc = judge_scores(JUDGE_PROMPT.format(query=query, pred=pred))
    labs = list(sc.keys())
    logps = torch.tensor([sc[lab] for lab in labs])
    probs = torch.softmax(logps, dim=0).numpy()
    samples = torch.multinomial(torch.tensor(probs, dtype=torch.float), k, replacement=True).tolist()
    votes = Counter(samples)
    top = max(votes, key=votes.get)
    return labs[top]

majority_fa = 0
for sentence, gold, pred in rule_results:
    verdict = judge_majority(sentence, pred)
    if verdict == "맞음" and gold != pred:
        majority_fa += 1

print(f"judge 판정 1회 때 false acceptance: {false_accept_count}건")
print(f"judge 판정 k={J_K}번 과반 투표 때 false acceptance: {majority_fa}건")
print("-> k를 늘려 투표하면 우연에 의한 승인은 줄어든다. 그러나 구조적으로 그럴듯한 오답은 그대로 통과한다")

## 4. 확인 질문

1. 규칙 채점과 judge 채점의 불일치는 몇 건이었나요? judge가 오답을 승인한 사례에서 judge는 무엇을 보고 "맞음"이라고 판정했나요?
2. pass@k 곡선에서 k를 늘릴 때 곡선이 어디서 천장에 닿았나요? 학습 전 pass@1과 대조해 무엇을 알 수 있나요?
3. judge 판정을 k번 뽑아 과반 투표하면 false acceptance가 줄었나요? 그래도 남는 것은 무엇 때문인가요?

답은 아래 셀에 글로 적으면 됩니다. 코드가 아니어도 됩니다.

*(여기에 답을 적으세요)*

## 5. 제출

1. 상단 메뉴 **파일 > .ipynb 다운로드** 로 이 노트북을 내려받습니다
2. [저장소](https://github.com/halla-ai/deepnlp-2026)의 `assignments/week-05/<내 학번>/` 에 업로드합니다
3. Pull Request를 엽니다

자세한 방법은 강의 사이트의 **과제 제출** 문서에 있습니다.

---

**막혔나요?** 오류 메시지의 마지막 줄을 먼저 읽어 보세요. 그래도 안 되면 AI Professor 튜터에게 묻고, 그래도 막히면 저장소 Issues에 남기세요.